# 05 - Risk scores, Prosecution Priority Score and baseline scenarios (SPEC sections 8.1-8.3)
* `risk_score`: **calibrated** P(collapse within 12 months) from the `champion` 12-month model (calibration from `collapse_calibration`, fitted in 04); P(24 months) = 1 − (1 − P12)² (constant annual hazard), because 24-month outcomes are not observable in the public register. Companies that have already collapsed are not scored: `risk_status = 'Collapsed'` with `collapse_status` and `collapse_date`. Otherwise with a band (Low / Elevated / High / Critical), the top 3 drivers, and the model version. Drivers from AUSTRAC features are flagged `restricted` so the app can redact them.
* `risk_score_history`: the same scores at each quarterly as-of snapshot (drives sparklines and Portfolio "movers").
* `scoring_weights` and `pps_params`: versioned criterion weights and **every** PPS constant (including the fitted P(success) coefficients), so a score can be reproduced exactly - by SQL and by the app's live what-if engine.
* SQL function `pps_compute(...)`: one transparent PPS formula; its constants default to the values in `pps_params`.
* `pps_score` (current) and `pps_score_history` (per as-of date): PPS per open case with the exact inputs used.
* `scenario`: a baseline plus one scenario per intervention preset for each case.

In [ ]:
%pip install -q shap scikit-learn mlflow
dbutils.library.restartPython()

In [ ]:
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "")
CATALOG, SCHEMA = dbutils.widgets.get("catalog"), dbutils.widgets.get("schema")
assert CATALOG and SCHEMA
FQ = f"{CATALOG}.{SCHEMA}"
WEIGHTS_VERSION = "spec-default-v1"
FEATURES = ["company_age_years", "is_licensee", "ato_debt_latest_aud", "ato_debt_trend_6m_pct", "lodgement_on_time_pct",
            "stp_headcount_trend_pct", "gst_cancelled", "smr_count_24m", "large_flow_smr_count_24m", "apra_flag_count",
            "afca_complaints_latest", "afca_complaints_trend_pct", "breach_reports_12m", "enforcement_alternatives_count",
            "banned_director_count", "director_prior_failed_companies", "related_collapsed_entities"]
LABELS = {"company_age_years": "Company age", "is_licensee": "Holds an ASIC licence", "ato_debt_latest_aud": "ATO debt",
          "ato_debt_trend_6m_pct": "ATO debt trend (6 months)", "lodgement_on_time_pct": "On-time lodgement rate",
          "stp_headcount_trend_pct": "Payroll headcount trend", "gst_cancelled": "GST registration cancelled",
          "smr_count_24m": "AUSTRAC SMRs (24 months)", "large_flow_smr_count_24m": "Large-flow SMRs",
          "apra_flag_count": "APRA prudential flags", "afca_complaints_latest": "AFCA complaints",
          "afca_complaints_trend_pct": "AFCA complaint trend", "breach_reports_12m": "Breach reports (12 months)",
          "enforcement_alternatives_count": "Prior EUs or infringement notices", "banned_director_count": "Directors on the banned register",
          "director_prior_failed_companies": "Directors with prior failed companies", "related_collapsed_entities": "Related entities that collapsed"}
AGENCY = {"ato_debt_latest_aud": "ATO", "ato_debt_trend_6m_pct": "ATO", "lodgement_on_time_pct": "ASIC", "stp_headcount_trend_pct": "ATO",
          "gst_cancelled": "ABR", "smr_count_24m": "AUSTRAC", "large_flow_smr_count_24m": "AUSTRAC", "apra_flag_count": "APRA",
          "afca_complaints_latest": "AFCA", "afca_complaints_trend_pct": "AFCA"}
RESTRICTED = {f for f, a in AGENCY.items() if a == "AUSTRAC"}

In [ ]:
import json, hashlib, mlflow, numpy as np, pandas as pd, shap
from pyspark.sql import functions as F, Window as W
mlflow.set_registry_uri("databricks-uc")
client = mlflow.MlflowClient()

# Only the 12-month model is served. The public register holds ~12 months of deregistrations, so 24-month outcomes are barely
# observable and the 24-month model does not validate out of time (it rarely passes the champion gate). P(24m) is derived from
# the calibrated 12-month probability under a constant annual hazard: P24 = 1 - (1 - P12)^2, which is also never below P12.
models, versions, calib = {}, {}, {}
for h in (12,):
    mv = client.get_model_version_by_alias(f"{FQ}.collapse_model_{h}m", "champion")
    versions[h] = f"collapse_model_{h}m/v{mv.version}"
    models[h] = mlflow.sklearn.load_model(f"models:/{FQ}.collapse_model_{h}m@champion")
    # Calibration fitted by 04 for this exact model version (Platt on log-odds, re-weighted to the observed population rate)
    row = (spark.table(f"{FQ}.collapse_calibration").filter((F.col("horizon_months") == h) & (F.col("model_version") == int(mv.version)))
                .orderBy(F.desc("created_at")).first())
    assert row is not None, f"no calibration for collapse_model_{h}m v{mv.version}; rerun 04_train_collapse_model"
    calib[h] = (row.calib_a, row.calib_b)
    print(f"{h}m v{mv.version}: calibrated to population rate {row.population_rate:.4f}, ECE {row.ece_raw:.3f} -> {row.ece_calibrated:.3f}")
MODEL_VERSION = f"{versions[12]};24m=hazard"
feat_all = spark.table(f"{FQ}.gold_entity_features")
# TreeExplainer is exact for gradient-boosted trees and runs in seconds (permutation took ~1 row/s)
explainer = shap.TreeExplainer(models[12])  # path-dependent: exact for HistGradientBoosting; the interventional mode is not

def score(pdf):
    X = pdf[FEATURES].astype(float).fillna(0.0)
    out = pdf[["entity_id", "acn", "as_of_date"]].copy()
    a, b = calib[12]
    out["p_collapse_12m"] = 1 / (1 + np.exp(-(a * models[12].decision_function(X) + b)))
    out["p_collapse_24m"] = 1 - (1 - out["p_collapse_12m"]) ** 2
    sv = np.asarray(explainer.shap_values(X))
    sv = sv[..., 1] if sv.ndim == 3 else sv  # log-odds contributions toward collapse
    def drivers(i):
        top = np.argsort(-np.abs(sv[i]))[:3]
        return json.dumps([{"feature": FEATURES[j], "label": LABELS[FEATURES[j]], "agency": AGENCY.get(FEATURES[j], "ASIC"),
                            "restricted": FEATURES[j] in RESTRICTED, "value": float(X.iloc[i, j]),
                            "direction": "raises" if sv[i, j] > 0 else "lowers", "shap": round(float(sv[i, j]), 4)} for j in top])
    out["drivers_json"] = [drivers(i) for i in range(len(out))]
    out["band"] = pd.cut(out.p_collapse_12m, [-0.01, 0.15, 0.35, 0.6, 1.0], labels=["Low", "Elevated", "High", "Critical"]).astype(str)
    out["model_version"] = MODEL_VERSION
    out["risk_status"] = "Active"
    return out

def save(pdf, table):
    sdf = spark.createDataFrame(pdf)
    for col, typ in (("collapse_status", "string"), ("collapse_date", "date")):  # typed explicitly: may be entirely null
        sdf = sdf.withColumn(col, F.col(col).cast(typ) if col in pdf.columns else F.lit(None).cast(typ))
    (sdf.withColumn("scored_at", F.current_timestamp()).withColumn("provenance", F.lit("derived"))
        .write.mode("overwrite").option("overwriteSchema", True).saveAsTable(f"{FQ}.{table}"))

latest = (feat_all.withColumn("rn", F.row_number().over(W.partitionBy("entity_id").orderBy(F.desc("as_of_date"))))
          .filter("rn = 1").drop("rn").toPandas())
# Companies that have already collapsed get no forward-looking probability: they show the collapse itself (status and date).
# Their feature rows stop before the collapse, so scoring them would only replay the risk they carried just beforehand.
collapsed = spark.table(f"{FQ}.silver_company_collapse").filter("collapse_date <= current_date()").select("acn", "collapse_status", "collapse_date").toPandas()
cur = score(latest).merge(collapsed, on="acn", how="left")
gone = cur["collapse_date"].notna()
cur.loc[gone, ["p_collapse_12m", "p_collapse_24m"]] = np.nan
cur.loc[gone, "band"], cur.loc[gone, "risk_status"], cur.loc[gone, "drivers_json"] = "Collapsed", "Collapsed", "[]"
cur["collapse_date"] = [str(d)[:10] if pd.notna(d) else None for d in cur["collapse_date"]]
cur["collapse_status"] = [v if isinstance(v, str) else None for v in cur["collapse_status"]]
print(f"active scored: {(~gone).sum()}, already collapsed: {gone.sum()}, mean P(12m) active = {cur.loc[~gone, 'p_collapse_12m'].mean():.3f}")
save(cur, "risk_score")

# History at each full quarterly snapshot (as-of dates covering most of the slice), for sparklines and movers
snap_dates = [r.as_of_date for r in feat_all.groupBy("as_of_date").count().filter("count >= 1000").orderBy("as_of_date").collect()]
hist = pd.concat([score(feat_all.filter(F.col("as_of_date") == d).toPandas()) for d in snap_dates], ignore_index=True)  # feature rows always precede any collapse
save(hist, "risk_score_history")
print("history as-of dates:", snap_dates)
display(spark.table(f"{FQ}.risk_score").groupBy("band").count())

In [ ]:
from sklearn.linear_model import LogisticRegression
hm = spark.table(f"{FQ}.silver_historical_matters").filter("success IS NOT NULL AND evidence_strength IS NOT NULL").toPandas()
lr = LogisticRegression().fit(hm[["evidence_strength"]].astype(float), hm["success"].astype(int))
B0, B1 = round(float(lr.intercept_[0]), 6), round(float(lr.coef_[0][0]), 6)

# Every constant in the PPS (SPEC 8.2). The app's live what-if engine reads this table, so SQL and UI agree exactly.
PARAMS = {
    "success_b0": B0, "success_b1": B1, "success_default_strength": 0.5, "success_fit_n": float(len(hm)),
    "fin_return_cap": 3.0,            # expected recovery / cost ratio that scores 100
    "collapse_clamp": 0.999,          # upper clamp on P(collapse 24m) before horizon scaling
    "harm_consumers_log_scale": 5.0,  # log10(consumers + 1) / 5 -> 100k consumers scores 1
    "harm_loss_log_scale": 8.0,       # log10(loss + 1) / 8 -> $100m loss scores 1
    "harm_w_consumers": 0.4, "harm_w_loss": 0.4, "harm_w_vulnerable": 0.2,
    "urg_w_collapse": 0.6, "urg_w_smr": 0.2, "urg_w_limitation": 0.2,
    "limitation_window_months": 60.0, "limitation_ramp_months": 48.0,
    "default_p_collapse_12m": 0.1, "default_p_collapse_24m": 0.2,
    "band_proceed": 70.0, "band_conditional": 50.0, "band_alternatives": 35.0,
}
PARAMS_VERSION = "pps-" + hashlib.sha256(json.dumps(PARAMS, sort_keys=True).encode()).hexdigest()[:10]
print(PARAMS_VERSION, f"P(success) = sigmoid({B0} + {B1} * evidence_strength), n={len(hm)}")

spark.createDataFrame([(WEIGHTS_VERSION, 0.25, 0.20, 0.20, 0.20, 0.15, "SPEC 8.2 defaults - ASIC to confirm")],
    "weights_version string, w_financial double, w_success double, w_harm double, w_deterrence double, w_urgency double, note string") \
    .withColumn("created_at", F.current_timestamp()).write.mode("overwrite").option("overwriteSchema", True).saveAsTable(f"{FQ}.scoring_weights")
spark.createDataFrame([(PARAMS_VERSION, WEIGHTS_VERSION, json.dumps(PARAMS), MODEL_VERSION)],
    "params_version string, weights_version string, params_json string, model_version string") \
    .withColumn("created_at", F.current_timestamp()).write.mode("overwrite").option("overwriteSchema", True).saveAsTable(f"{FQ}.pps_params")

# Each preset is a bundle of multipliers / overrides applied to the case inputs
spark.createDataFrame([
    ("asset_freeze_first", "Seek asset freeze first", 1.10, 1.0, 0.20, None, 1.0),
    ("enforceable_undertaking", "Enforceable undertaking instead", 0.25, 0.6, None, 0.90, 0.5),
    ("litigation_cost_up_40", "Litigation cost +40%", 1.40, 1.0, None, None, 1.0),
    ("expedited_hearing", "Expedited hearing", 0.90, 0.6, None, None, 1.0)],
    "preset_id string, name string, cost_multiplier double, duration_multiplier double, p_collapse_override double, p_success_override double, recoverable_multiplier double") \
    .write.mode("overwrite").saveAsTable(f"{FQ}.intervention_presets")

In [ ]:
P = PARAMS
# Criteria are normalised to 0-100. Bands compare the ROUNDED total, so the displayed score always matches its band.
spark.sql(f'''
CREATE OR REPLACE FUNCTION {FQ}.pps_compute(
  recoverable DOUBLE, litigation_cost DOUBLE, duration_months DOUBLE, p_success DOUBLE, p_collapse_12m DOUBLE, p_collapse_24m DOUBLE,
  consumers_affected DOUBLE, consumer_loss DOUBLE, vulnerable BOOLEAN, strategic_priority DOUBLE, months_to_limitation DOUBLE, recent_smr BOOLEAN,
  w_financial DOUBLE, w_success DOUBLE, w_harm DOUBLE, w_deterrence DOUBLE, w_urgency DOUBLE)
RETURNS STRUCT<total: DOUBLE, band: STRING, financial: DOUBLE, success: DOUBLE, harm: DOUBLE, deterrence: DOUBLE, urgency: DOUBLE,
               p_collapse_before_judgment: DOUBLE, expected_recovery: DOUBLE, expected_net_recovery: DOUBLE, return_ratio: DOUBLE>
COMMENT 'Prosecution Priority Score (SPEC 8.2), params {PARAMS_VERSION}. A decision aid only; it never makes the decision.'
RETURN (
  SELECT named_struct(
    'total', total,
    'band', CASE WHEN total >= {P["band_proceed"]} THEN 'Proceed' WHEN total >= {P["band_conditional"]} THEN 'Proceed – conditional'
                 WHEN total >= {P["band_alternatives"]} THEN 'Consider alternatives' ELSE 'Do not pursue' END,
    'financial', round(fin, 1), 'success', round(suc, 1), 'harm', round(hrm, 1), 'deterrence', round(det, 1), 'urgency', round(urg, 1),
    'p_collapse_before_judgment', round(pcb, 4), 'expected_recovery', round(er, 0), 'expected_net_recovery', round(er - litigation_cost, 0),
    'return_ratio', round(er / nullif(litigation_cost, 0), 3))
  FROM (SELECT *, round(w_financial * fin + w_success * suc + w_harm * hrm + w_deterrence * det + w_urgency * urg, 1) AS total
  FROM (SELECT pcb, er,
    100 * least(coalesce(er / nullif(litigation_cost, 0), 0) / {P["fin_return_cap"]}, 1.0) AS fin,
    100 * p_success AS suc,
    100 * ({P["harm_w_consumers"]} * least(log10(greatest(consumers_affected, 0) + 1) / {P["harm_consumers_log_scale"]}, 1)
         + {P["harm_w_loss"]} * least(log10(greatest(consumer_loss, 0) + 1) / {P["harm_loss_log_scale"]}, 1)
         + {P["harm_w_vulnerable"]} * CAST(coalesce(vulnerable, false) AS INT)) AS hrm,
    least(greatest(coalesce(strategic_priority, 0), 0), 100) AS det,
    100 * ({P["urg_w_collapse"]} * p_collapse_12m + {P["urg_w_smr"]} * CAST(coalesce(recent_smr, false) AS INT)
         + {P["urg_w_limitation"]} * least(greatest(({P["limitation_window_months"]} - months_to_limitation) / {P["limitation_ramp_months"]}, 0), 1)) AS urg
    FROM (SELECT pcb, recoverable * p_success * (1 - pcb) AS er
          FROM (SELECT 1 - pow(1 - least(greatest(p_collapse_24m, 0), {P["collapse_clamp"]}), duration_months / 24.0) AS pcb))))
)''')

def case_base(as_of_sql, risk_table, view):
    # Inputs as known on as_of: risk at that date, limitation months from that date, cost to date from cumulative spend
    spark.sql(f'''
    CREATE OR REPLACE TEMP VIEW {view} AS
    SELECT c.* EXCEPT (cost_to_date_aud), coalesce(sp.cost_to_date_aud, 0) AS cost_to_date_aud, w.w_financial, w.w_success, w.w_harm,
           w.w_deterrence, w.w_urgency, w.weights_version, r.model_version,
           -- A collapsed respondent has already failed: recovery is discounted as if collapse is certain
           CASE WHEN r.risk_status = 'Collapsed' THEN 1.0 ELSE coalesce(r.p_collapse_12m, {P["default_p_collapse_12m"]}) END AS p12,
           CASE WHEN r.risk_status = 'Collapsed' THEN 1.0 ELSE coalesce(r.p_collapse_24m, {P["default_p_collapse_24m"]}) END AS p24,
           r.entity_id IS NULL AS risk_defaulted,
           coalesce(c.counsel_evidence_strength, c.evidence_element_coverage, {P["success_default_strength"]}) AS evidence_strength_used,
           1 / (1 + exp(-({P["success_b0"]} + {P["success_b1"]} * coalesce(c.counsel_evidence_strength, c.evidence_element_coverage, {P["success_default_strength"]})))) AS p_success_base,
           round(months_between(c.limitation_date, {as_of_sql}), 2) AS months_to_limitation, {as_of_sql} AS as_of_date
    FROM {FQ}.gold_case_inputs c
    LEFT JOIN {risk_table} r ON r.entity_id = c.respondent_entity_id
    LEFT JOIN (SELECT case_id, sum(spend_aud) AS cost_to_date_aud FROM {FQ}.gold_case_spend WHERE month <= {as_of_sql} GROUP BY case_id) sp
           ON sp.case_id = c.case_id
    CROSS JOIN {FQ}.scoring_weights w
    WHERE w.weights_version = '{WEIGHTS_VERSION}' AND to_date(c.opened_at) <= {as_of_sql}''')

PPS_ARGS = '''est_recoverable_aud, est_litigation_cost_aud, est_duration_months, p_success_base, p12, p24, consumers_affected,
              total_consumer_loss_aud, vulnerable_cohort, strategic_priority_score, months_to_limitation, recent_smr_flag,
              w_financial, w_success, w_harm, w_deterrence, w_urgency'''
INPUTS_JSON = '''to_json(named_struct('recoverable', est_recoverable_aud, 'litigation_cost', est_litigation_cost_aud,
        'duration_months', est_duration_months, 'p_success', p_success_base, 'evidence_strength', evidence_strength_used,
        'p_collapse_12m', p12, 'p_collapse_24m', p24, 'consumers_affected', consumers_affected, 'consumer_loss', total_consumer_loss_aud,
        'vulnerable', vulnerable_cohort, 'strategic_priority', strategic_priority_score, 'months_to_limitation', months_to_limitation,
        'recent_smr', recent_smr_flag, 'cost_to_date', cost_to_date_aud))'''

def pps_select(view):
    return f'''SELECT case_id, as_of_date, weights_version, '{PARAMS_VERSION}' AS params_version, model_version, s.total, s.band,
       to_json(s) AS criteria_json, {INPUTS_JSON} AS inputs_json, p_success_base AS p_success, p12 AS p_collapse_12m, p24 AS p_collapse_24m,
       risk_defaulted, cost_to_date_aud, current_timestamp() AS scored_at, 'derived' AS provenance
       FROM (SELECT *, {FQ}.pps_compute({PPS_ARGS}) AS s FROM {view})'''

case_base("current_date()", f"{FQ}.risk_score", "case_base")
spark.sql(f"CREATE OR REPLACE TABLE {FQ}.pps_score AS {pps_select('case_base')}")

# History: PPS at each quarterly as-of snapshot, with risk as scored at that date
hist_sql = []
for i, d in enumerate(snap_dates):
    v = f"case_base_h{i}"
    case_base(f"date'{d}'", f"(SELECT * FROM {FQ}.risk_score_history WHERE as_of_date = date'{d}')", v)
    hist_sql.append(pps_select(v))
hist_sql.append(pps_select("case_base"))
spark.sql(f"CREATE OR REPLACE TABLE {FQ}.pps_score_history AS " + " UNION ALL ".join(hist_sql))

In [ ]:
# Baseline plus one scenario per intervention preset, stamped with inputs, author and versions so it can be reproduced
spark.sql(f'''
CREATE OR REPLACE TABLE {FQ}.scenario AS
WITH inp AS (
  SELECT b.*, 'Baseline' AS name, CAST(NULL AS STRING) AS preset_id, est_recoverable_aud AS rec, est_litigation_cost_aud AS cost,
         est_duration_months AS dur, p_success_base AS ps, p12 AS sp12, p24 AS sp24
  FROM case_base b
  UNION ALL
  SELECT b.*, p.name, p.preset_id, est_recoverable_aud * p.recoverable_multiplier, est_litigation_cost_aud * p.cost_multiplier,
         est_duration_months * p.duration_multiplier, coalesce(p.p_success_override, p_success_base),
         coalesce(p.p_collapse_override, p12), coalesce(p.p_collapse_override, p24)
  FROM case_base b CROSS JOIN {FQ}.intervention_presets p)
SELECT sha2(concat_ws('|', case_id, coalesce(preset_id, 'baseline'), '{PARAMS_VERSION}'), 256) AS scenario_id, case_id, name, preset_id,
       'system' AS author,
       to_json(named_struct('recoverable', rec, 'litigation_cost', cost, 'duration_months', dur, 'p_success', ps,
               'p_collapse_12m', sp12, 'p_collapse_24m', sp24, 'consumers_affected', consumers_affected, 'consumer_loss', total_consumer_loss_aud,
               'vulnerable', vulnerable_cohort, 'strategic_priority', strategic_priority_score, 'months_to_limitation', months_to_limitation,
               'recent_smr', recent_smr_flag)) AS inputs_json,
       to_json({FQ}.pps_compute(rec, cost, dur, ps, sp12, sp24, consumers_affected, total_consumer_loss_aud, vulnerable_cohort,
               strategic_priority_score, months_to_limitation, recent_smr_flag, w_financial, w_success, w_harm, w_deterrence, w_urgency)) AS outputs_json,
       model_version, weights_version, '{PARAMS_VERSION}' AS params_version, as_of_date, current_timestamp() AS created_at
FROM inp''')
display(spark.sql(f"SELECT band, count(*) n, round(avg(total),1) avg_pps FROM {FQ}.pps_score GROUP BY band"))
display(spark.sql(f"SELECT as_of_date, count(*) cases, round(avg(total),1) avg_pps FROM {FQ}.pps_score_history GROUP BY 1 ORDER BY 1"))